# P11 · Eficiencia del mercado y detección de anomalías
**QuantLab | Módulo 1 · Fundamentos financieros | Práctica 11 de 12**

**Pregunta:** ¿cómo distinguir información predictiva de azar y sobreajuste?

Trabajaremos con un control nulo (retornos independientes), un control con autocorrelación y un mercado artificial cuya dependencia desaparece. Después podrás importar velas reales desde P04. Ninguna simulación demuestra una ineficiencia de BTC.

Al terminar podrás calcular autocorrelaciones, interpretar p-valores, corregir pruebas múltiples, congelar hipótesis antes de evaluar el test y diferenciar significación estadística de viabilidad económica. Los ejercicios están resueltos en la sección 8, sin modificar el test reservado.

Las funciones están en **`src/quantlab/p11/efficiency.py`** y sus pruebas en **`tests/test_efficiency.py`** (`pytest` desde la raíz del proyecto).

### Resultados de referencia (ejecución sintética guardada)
- 20 de 500 indicadores nulos superaron el umbral nominal del 5 %; ninguno sobrevivió a Bonferroni o Benjamini–Hochberg.
- El IC orientado del candidato elegido cayó de 0,0756 en desarrollo a 0,0209 en validación (p = 0,5460).
- En el mercado con decaimiento, la autocorrelación de retardo 1 pasó de 0,4087 a −0,0107.
- Incluso en el control persistente, la operación media de validación pasó de +0,193 bps brutos a −19,807 bps netos con el coste hipotético de 20 bps.

**Al cambiar la configuración, reinterpreta estos valores utilizando las salidas actualizadas.** Estas cifras no son resultados empíricos de Bitcoin.

## 1. Configuración y contrato de datos
Python con NumPy, Pandas, SciPy, statsmodels y Matplotlib. En Colab, si falta una dependencia, ejecuta `%pip install statsmodels scipy`. En VS Code selecciona el kernel QuantLab. Todos los tiempos están en UTC y corresponden al cierre del intervalo; las rentabilidades son decimales, salvo donde se indique bps (1 bps = 0,01 %).

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from IPython.display import display
import inspect
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p11.efficiency import (bonferroni_manual, diagnostics, false_positive_table, ljung_box_power,
                                     ljung_box_table, load_p04, longest_valid_run, make_market, make_markets,
                                     manual_autocorrelation, max_ic_by_count, monte_carlo_false_positives,
                                     null_signal_experiment, pearson_many, random_walk_acf, regime_detection_delay,
                                     rolling_autocorr, split_returns, trade_experiment)

SEED = 42
N = 2400
WINDOW = '1min'
TRAIN_FRACTION = 0.65
LAGS = 10
ALPHA = 0.05
N_SIGNALS = 500
N_SIMULATIONS = 3000
ROUND_TRIP_BPS = 20.0  # supuesto didáctico, no tarifa real
DATA_MODE = 'synthetic'  # 'synthetic' o 'csv'
CSV_PATH = 'quantlab_exports/p04_live_REEMPLAZAR_series.csv'
EXPORT = False
plt.rcParams.update({'figure.figsize': (9, 3.3), 'figure.dpi': 110,
                     'axes.grid': True, 'grid.alpha': .2, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
assert 0 < TRAIN_FRACTION < 1 and 0 < ALPHA < 1 and LAGS >= 1

## 2. Mercado eficiente no significa precio constante
La forma débil de la hipótesis de eficiencia plantea que la información contenida en precios históricos no permite obtener beneficios anormales ajustados por riesgo y costes de forma sistemática. Un paseo aleatorio en log-precios es un **modelo nulo simplificado**, no una equivalencia de la eficiencia.

$$\log P_t=\log P_{t-1}+r_t,\qquad r_t=\phi_t r_{t-1}+\varepsilon_t$$

- $\phi=0$: innovaciones independientes (control negativo).
- $\phi=0.35$: persistencia (control positivo).
- $\phi$ pasa de 0.35 a 0: desaparición artificial de una señal.

Usamos el mismo ruido para comparar los tres mecanismos. El corte se fija de antemano al 65 %; no depende de resultados.

In [ ]:
print(inspect.getsource(make_market))

markets = make_markets(N, SEED, TRAIN_FRACTION, WINDOW)
if DATA_MODE == 'csv':
    markets['CSV P04'] = load_p04(CSV_PATH, WINDOW)  # valida rejilla y precios (reutiliza P05)
elif DATA_MODE != 'synthetic':
    raise ValueError('DATA_MODE no reconocido')
print('Fuentes: simulaciones seed', SEED, '| CSV adicional:', CSV_PATH if DATA_MODE=='csv' else 'no')
display(pd.DataFrame([{'serie': k, 'velas':len(v), 'huecos':v.isna().sum(),'inicio':v.index.min(),'fin':v.index.max()} for k,v in markets.items()]))

In [ ]:
fig, ax = plt.subplots()
market_colors = dict(zip(markets, plt.rcParams['axes.prop_cycle'].by_key()['color']))
for name, price in markets.items():
    ax.plot(price.index, price/price.dropna().iloc[0]*100, label=name, lw=1.2, color=market_colors[name])
ax.set(title='Trayectorias de precios: base 100', ylabel='Índice de precio', xlabel='Cierre UTC')
ax.legend(); fig.autofmt_xdate(); plt.show()

## 3. Autocorrelación y Ljung–Box
$$\rho_k=\operatorname{Corr}(r_t,r_{t-k})$$
Una autocorrelación positiva sugiere persistencia lineal; negativa, reversión. La probabilidad de acertar la dirección no se obtiene directamente de esta magnitud.

Ljung–Box contrasta conjuntamente que las primeras $k$ autocorrelaciones son cero. Un p-valor bajo permite rechazar esa hipótesis bajo los supuestos del contraste; no es la probabilidad de que el mercado sea eficiente. En datos reales con heterocedasticidad, dependencia no lineal o cambios de régimen, los p-valores clásicos pueden estar mal calibrados. Usaremos el tramo contiguo más largo de cada partición para no comprimir huecos. Exigimos al menos $5k+1$ retornos y un mínimo de 60; para inferencia real se necesitarán pruebas robustas y más histórico.

In [ ]:
print(inspect.getsource(split_returns))
print(inspect.getsource(diagnostics))

diagnostics_table, acf_table = diagnostics(markets, LAGS, TRAIN_FRACTION)
display(diagnostics_table.round(5))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,3.6),sharey=True)
for ax, period in zip(axes,['Desarrollo','Validación']):
    for name, group in acf_table[acf_table.periodo==period].groupby('mercado'):
        ax.plot(group.lag,group.acf,marker='o',ms=3,label=name,color=market_colors[name])
    ax.axhline(0,color='black',lw=.8)
    ax.set(title=period,xlabel='Retardo (velas)',xticks=range(1,LAGS+1))
axes[0].set_ylabel('Autocorrelación de retornos')
axes[1].legend(fontsize=8); fig.tight_layout(); plt.show()

**Interpreta:** compara el control aleatorio con el persistente. En Decaimiento observa si el patrón de desarrollo sobrevive en validación. No ajustes los retardos al ver el test. No confundas autocorrelación de precios (habitualmente alta incluso en un paseo aleatorio) con autocorrelación de retornos.

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Los controles se comportan como se esperaba</b>

| Mercado | ACF(1) desarrollo | Ljung–Box (10) | ACF(1) validación | Ljung–Box (10) |
|---|---|---|---|---|
| Aleatorio | +0,071 | p = 0,13 | −0,011 | p = 0,64 |
| Persistente | **+0,409** | p ≈ 0 | **+0,340** | p ≈ 0 |
| Decaimiento | **+0,409** | p ≈ 0 | **−0,011** | p = 0,64 |

- **Persistente:** la dependencia es clara y se mantiene en los dos periodos.
- **Decaimiento:** idéntico al persistente en desarrollo y al aleatorio en validación. El patrón **no sobrevive**: quien lo hubiera descubierto en desarrollo habría perdido la señal justo al usarla.
- **Aleatorio:** su ACF(1) de desarrollo (0,071) es casi 3 errores estándar (1/√1.559 ≈ 0,025). Un solo retardo puede parecer "significativo" por azar. Ljung–Box, que mira los 10 retardos a la vez, no lo rechaza (p = 0,13).

</div>

## 4. Pruebas múltiples y Data Snooping
Un investigador prueba 500 señales independientes, generadas al azar antes de observar los retornos. Cada prueba contrasta $H_0:\rho=0$ mediante Pearson. Utilizamos el control nulo, con retornos IID normales e indicadores independientes, para que el contraste clásico sea apropiado. No hacemos esta prueba ingenua con indicadores reales solapados.

$$E[V]=m\alpha,\quad P(V\geq1)=1-(1-\alpha)^m$$

Bonferroni controla la probabilidad de al menos un falso rechazo (FWER) con umbral $\alpha/m$. Benjamini–Hochberg controla la tasa esperada de falsos descubrimientos (FDR) bajo independencia o ciertas dependencias positivas. Ninguno sustituye un test temporal reservado.

In [ ]:
print(inspect.getsource(pearson_many))
print(inspect.getsource(null_signal_experiment))

r = split_returns(markets['Aleatorio'], TRAIN_FRACTION)
y_dev = r['Desarrollo'].dropna().to_numpy()
y_test = r['Validación'].dropna().to_numpy()
signal_tests, best, direction = null_signal_experiment(y_dev, y_test, N_SIGNALS, ALPHA, seed=SEED+10)
ic_dev, p_dev, ic_test, p_test = (signal_tests[c].to_numpy() for c in ['ic_dev', 'p_dev', 'ic_test', 'p_test'])
reject_bonf, reject_fdr = signal_tests.p_bonferroni < ALPHA, signal_tests.p_bh < ALPHA
print(f'Falsos positivos sin corregir: {(p_dev<ALPHA).sum()} / {N_SIGNALS}; esperados: {N_SIGNALS*ALPHA:g}')
print(f'Bonferroni: {reject_bonf.sum()} | BH: {reject_fdr.sum()}')
print(f'Candidato congelado: {best} | IC orientado desarrollo: {direction*ic_dev[best]:.4f} | validación: {direction*ic_test[best]:.4f} | p test: {p_test[best]:.4f}')
display(signal_tests.loc[[best]].round(5))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,3.6))
axes[0].hist(p_dev,bins=np.linspace(0,1,21),edgecolor='white')
axes[0].axvline(ALPHA,color='firebrick',ls='--',label='Umbral 5 %' if ALPHA==.05 else 'Umbral alfa')
axes[0].set(title='P-valores de 500 señales nulas' if N_SIGNALS==500 else f'P-valores de {N_SIGNALS} señales nulas',xlabel='p-valor',ylabel='Número de señales'); axes[0].legend()
axes[1].scatter(ic_dev,ic_test,s=9,alpha=.45,label='Señales nulas')
axes[1].scatter(ic_dev[best],ic_test[best],c='firebrick',s=60,marker='*',label='Elegida en desarrollo')
axes[1].axvline(0,color='grey',lw=.7); axes[1].axhline(0,color='grey',lw=.7)
axes[1].set(title='Descubrimiento frente a validación',xlabel='IC desarrollo',ylabel='IC validación'); axes[1].legend(fontsize=8)
fig.tight_layout(); plt.show()

## 5. Monte Carlo: repetimos el experimento miles de veces
Bajo la hipótesis nula ideal, los p-valores independientes tienen distribución uniforme. Simulamos directamente esos p-valores (equivalente para contar rechazos bajo estos supuestos; no estamos simulando estrategias de mercado).

In [ ]:
print(inspect.getsource(monte_carlo_false_positives))

mc = monte_carlo_false_positives(N_SIMULATIONS, N_SIGNALS, ALPHA, seed=SEED+20)
false_counts = mc['false_counts']
print(f"Media de falsos positivos: {mc['mean']:.2f}; teoría: {mc['theory']:.2f}")
print(f"Probabilidad de >=1 sin corregir: {mc['p_at_least_one']:.2%}; Bonferroni: {mc['p_at_least_one_bonferroni']:.2%}")
fig, ax = plt.subplots()
ax.hist(false_counts,bins=np.arange(false_counts.min()-.5,false_counts.max()+1.5),edgecolor='white')
ax.axvline(N_SIGNALS*ALPHA,color='firebrick',ls='--',label='Esperanza teórica')
ax.set(title=f'Falsos positivos por experimento ({N_SIMULATIONS:,} simulaciones)',xlabel='Número de rechazos falsos',ylabel='Frecuencia'); ax.legend(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Probar mucho garantiza encontrar algo, aunque no haya nada</b>

- **20 de 500 señales nulas** tienen p < 0,05 en desarrollo (se esperaban 25). Ninguna sobrevive a Bonferroni ni a Benjamini–Hochberg.
- La "mejor" señal (la 105) tiene un IC de 0,076 en desarrollo y p = 0,003, que parecería un hallazgo. En validación cae a **0,021** (p = 0,55): era azar.
- **Monte Carlo:** de media salen 25,15 falsos positivos por experimento (teoría: 25), y en el **100 %** de los experimentos aparece al menos uno. Con Bonferroni, esa probabilidad baja al **4,7 %**, cerca del 5 % que se quería controlar.

</div>

## 6. Del patrón estadístico al coste económico
La señal de persistencia se fija antes del experimento: $s_t=\operatorname{sign}(r_t)$. Se simula una posición spot 0/1: compra si el último retorno es positivo, no vende en corto. La señal se conoce al cierre $t$, entrada teórica al cierre $t+1$, salida al cierre $t+2$. Usamos decisiones cada 2 velas para que no se solapen operaciones; no se mantienen posiciones entre ellas. Sin libro de órdenes, los cierres son referencias y el coste round-trip es una aproximación explícita, no ejecutabilidad demostrada. Se descartan huecos y ventanas que cruzan el corte.

$$R_{neto}=P_{t+2}/P_{t+1}-1-c$$
Comparamos el retorno medio por operación, no el rendimiento compuesto de una cartera. El coste incluye ambas piernas; no se resta dos veces.

In [ ]:
print(inspect.getsource(trade_experiment))

performance = pd.concat([trade_experiment(p,ROUND_TRIP_BPS,TRAIN_FRACTION).assign(mercado=name) for name,p in markets.items()],ignore_index=True)
display(performance.round(3))
fig, ax = plt.subplots(figsize=(9,3.6))
valid = performance[performance.periodo=='Validación'].set_index('mercado')
valid[['gross_bps','net_bps']].rename(columns={'gross_bps':'Bruto','net_bps':'Neto'}).plot.bar(ax=ax,rot=0)
ax.axhline(0,color='black',lw=.8); ax.set(title='Validación: retorno medio por operación',xlabel='',ylabel='bps'); fig.tight_layout(); plt.show()

## 7. Controles de corrección
Los tests validan código y convenciones, no prueban rentabilidad financiera. Los controles deterministas deben funcionar aunque cambies el origen de datos.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: tamaños, Pearson frente a SciPy, p-valores, correcciones, huecos y costes
assert len(markets['Aleatorio']) == N
assert len(y_dev)+len(y_test)==N-2
assert np.isclose(pearson_many(np.array([[1],[2],[4],[3]]),np.array([1.,2.,3.,4.]))[0][0],stats.pearsonr([1,2,4,3],[1,2,3,4]).statistic)
assert np.all((p_dev>=0)&(p_dev<=1))
assert reject_bonf.sum() <= (p_dev<ALPHA).sum()
assert reject_fdr.sum() <= (p_dev<ALPHA).sum()
assert np.isclose(N_SIGNALS*ALPHA, (np.arange(N_SIGNALS)*0+ALPHA).sum())
assert len(longest_valid_run(pd.Series([1.,2.,np.nan,3.,4.,5.])))==3
assert np.allclose(performance.gross_bps-performance.net_bps,ROUND_TRIP_BPS,equal_nan=True)
assert np.isclose(trade_experiment(pd.Series(np.arange(100.,200.)),0).iloc[0].gross_bps,
                  trade_experiment(pd.Series(np.arange(100.,200.)),0).iloc[0].net_bps)
print('10 comprobaciones superadas.')

## 8. Ejercicios

**E1. Autocorrelación manual.** Implementa la correlación entre dos arrays desplazados sin `Series.autocorr`. Compara retardos 1, 2 y 5 con los resultados de la sección 3. Explica el signo.

**E2. Paseo aleatorio.** Simula 100 series independientes de precios; compara autocorrelaciones de niveles y de retornos. ¿Por qué un precio con tendencia aparente no demuestra predictibilidad?

**E3. Ljung–Box.** Aplica el contraste con retardos fijados a priori de 5 y 10. Explica qué significa rechazar y no rechazar. ¿Qué cambia al multiplicar el tamaño de muestra?

**E4. Falsos positivos.** Implementa Bonferroni sin librerías. Calcula el número esperado de falsos positivos para 20, 100 y 500 hipótesis; compara con Monte Carlo. Explica la diferencia entre FWER y FDR.

**E5. Data Snooping.** Selecciona el mejor IC en desarrollo de entre 100, 500 y 1.000 señales nulas. Para cada experimento nuevo usa una semilla nueva y congela la selección antes de evaluar. ¿Por qué crece el IC máximo incluso sin información predictiva?

**E6. Alpha Decay.** Construye una autocorrelación móvil de retardo 1 con ventana de 200 retornos, sin centrarla. Marca el corte temporal. ¿Se detecta inmediatamente un cambio de régimen? No uses esta visualización para reajustar el test ya observado.

**E7. Costes.** Repite la simulación pedagógica con costes de 0, 5, 10 y 40 bps. ¿Puede un patrón estadístico desaparecer económicamente? Explica la diferencia entre un coste hipotético y uno medido con P08.

**E8. Datos reales.** Exporta `series.csv` y metadatos desde P04 con `EXPORT=True`, usa un histórico real suficientemente largo e importa el fichero aquí. Ajusta WINDOW y comprueba su procedencia; reporta huecos, tamaño de cada partición, magnitud del efecto, costes y limitaciones. Un histórico muy corto debe conducir a «evidencia insuficiente», no a una conclusión categórica.

### E1 · Autocorrelación manual

In [ ]:
print(inspect.getsource(manual_autocorrelation))

my_autocorrelation = manual_autocorrelation
rows = []
for name, price in markets.items():
    s = longest_valid_run(split_returns(price, TRAIN_FRACTION)['Desarrollo'])
    for k in (1, 2, 5):
        rows.append({'mercado': name, 'lag': k, 'manual': my_autocorrelation(s.to_numpy(), k), 'pandas': s.autocorr(k)})
e1 = pd.DataFrame(rows)
assert np.allclose(e1.manual, e1.pandas)
display(e1.pivot(index='mercado', columns='lag', values='manual').round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · El signo dice persistencia o reversión</b>

Desarrollo:

| Mercado | Retardo 1 | Retardo 2 | Retardo 5 |
|---|---|---|---|
| Aleatorio | +0,071 | −0,003 | −0,003 |
| Persistente | **+0,409** | **+0,140** | −0,015 |
| Decaimiento | +0,409 | +0,140 | −0,015 |

- La implementación propia (pares x_t y x_{t−k} en los que ambos existen, Pearson explícito) coincide con `Series.autocorr`.
- En el persistente, el **signo positivo** indica persistencia: un retorno positivo tiende a ir seguido de otro positivo.
- La autocorrelación decae como φ^k: 0,35 → 0,12 → 0,005. Los valores observados (0,41, 0,14 y −0,015) siguen ese patrón con ruido de muestra. Con φ = 0,35, a 5 velas ya no queda nada.

</div>

### E2 · Paseos aleatorios: niveles frente a retornos

In [ ]:
print(inspect.getsource(random_walk_acf))

e2 = random_walk_acf(100, 1000, seed=7)
display(e2.describe().loc[['mean', 'std', 'min', 'max']].round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · Un precio sin memoria parece tener memoria</b>

100 paseos aleatorios independientes de 1.000 velas:

| | Media | Desv. típica | Mínimo | Máximo |
|---|---|---|---|---|
| ACF(1) de **niveles** | **0,995** | 0,004 | 0,980 | 0,999 |
| ACF(1) de **retornos** | **0,001** | 0,036 | −0,082 | 0,111 |
| Retorno total | −0,07 % | 3,1 % | −7,0 % | +8,0 % |

- Los **niveles** de precio tienen una autocorrelación de casi 1 aunque los retornos sean independientes: el precio de hoy es el de ayer más un poco de ruido. Eso no es predictibilidad, es acumulación.
- Los **retornos** tienen autocorrelación ≈ 0: no hay nada que predecir.
- **Tendencias aparentes:** algunas series suben un 8 % y otras caen un 7 % sin ninguna causa. Con muchas series, siempre habrá alguna con una "tendencia" bonita. Una tendencia en el gráfico de precios no prueba que el siguiente retorno se pueda anticipar.

</div>

### E3 · Ljung–Box con retardos fijados a priori

In [ ]:
print(inspect.getsource(ljung_box_power))

e3 = pd.concat({f'{m} · {p}': ljung_box_table(r, (5, 10)) for m in ['Aleatorio', 'Persistente']
                for p, r in split_returns(markets[m], TRAIN_FRACTION).items()})
display(e3.round(4))
print("Frecuencia de rechazo al 5 % con una dependencia débil (φ = 0,05), 200 repeticiones por tamaño:")
display(ljung_box_power(.05, (500, 2000, 10000), lags=5, reps=200))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Rechazar no es probar, y no rechazar no es demostrar</b>

| Serie | p (5 retardos) | p (10 retardos) |
|---|---|---|
| Aleatorio · desarrollo | 0,149 | 0,133 |
| Aleatorio · validación | 0,905 | 0,636 |
| Persistente · desarrollo | ≈ 0 | ≈ 0 |
| Persistente · validación | ≈ 0 | ≈ 0 |

- **Rechazar** (persistente): los datos son muy improbables si todas las autocorrelaciones fueran cero, así que hay alguna dependencia lineal. No dice cuánta, ni si es explotable.
- **No rechazar** (aleatorio): los datos son compatibles con autocorrelación cero, pero **no demuestran** que lo sea. Puede haber una dependencia demasiado débil para detectarla, o no lineal.
- **Tamaño de muestra:** con una dependencia débil fija (φ = 0,05), Ljung–Box la detecta el **11,5 %** de las veces con 500 datos, el **29,5 %** con 2.000 y el **98 %** con 10.000. Al multiplicar la muestra, efectos minúsculos se vuelven "significativos". Por eso un p-valor bajo no dice nada sobre el **tamaño** del efecto, que es lo que importa para ganar dinero.

</div>

### E4 · Bonferroni propio y falsos positivos

In [ ]:
print(inspect.getsource(bonferroni_manual))

mine = bonferroni_manual(p_dev, ALPHA)
assert (mine == reject_bonf.to_numpy()).all()
print(f"Bonferroni propio: umbral {ALPHA/len(p_dev):.6f} | rechazos {mine.sum()} (igual que statsmodels)")
display(false_positive_table((20, 100, 500), ALPHA, N_SIMULATIONS, SEED+20).round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · FWER frente a FDR</b>

| Hipótesis | Falsos positivos esperados | Monte Carlo | P(≥ 1) teórica | Monte Carlo | Con Bonferroni |
|---|---|---|---|---|---|
| 20 | 1,0 | 1,00 | 64,2 % | 64,3 % | 5,0 % |
| 100 | 5,0 | 5,09 | 99,4 % | 99,3 % | 5,1 % |
| 500 | 25,0 | 25,15 | 100 % | 100 % | 4,7 % |

- La implementación propia (rechazar si p < α / m) da los mismos rechazos que `statsmodels`: con 500 señales el umbral es 0,0001 y no rechaza ninguna.
- **FWER** (*family-wise error rate*), que controla Bonferroni, es la probabilidad de cometer **al menos un** falso positivo. Es muy estricto: con 500 pruebas exige p < 0,0001, y puede perder efectos reales.
- **FDR** (*false discovery rate*), que controla Benjamini–Hochberg, es la **proporción esperada de falsos** entre lo que se declara descubierto. Es menos estricto: acepta algunos falsos positivos a cambio de detectar más efectos reales. Sirve cuando se prueban muchas hipótesis y se puede tolerar que una parte de los "hallazgos" sea ruido.

</div>

### E5 · Data snooping con 100, 500 y 1.000 señales

In [ ]:
print(inspect.getsource(max_ic_by_count))

e5 = max_ic_by_count(y_dev, y_test, (100, 500, 1000), (101, 202, 303))  # una semilla nueva por experimento
display(e5.round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · Cuantas más señales se prueban, mejor parece la mejor</b>

| Señales probadas | IC máximo en desarrollo | Aproximación teórica | IC orientado en validación | p en validación |
|---|---|---|---|---|
| 100 | 0,069 | 0,077 | −0,002 | 0,96 |
| 500 | 0,078 | 0,089 | −0,009 | 0,79 |
| 1.000 | **0,092** | 0,094 | +0,025 | 0,46 |

- **El IC máximo crece con el número de señales** aunque ninguna tenga información: es el máximo de cada vez más correlaciones aleatorias. Con n observaciones, cada IC nulo tiene una desviación típica de ≈ 1/√n, y el máximo de m de ellas crece como √(2 · ln m / n).
- En validación, con la selección y el signo **congelados** antes de mirar, todos caen a ≈ 0.
- **Lección:** un IC alto solo significa algo si se sabe **cuántas** señales se probaron para encontrarlo.

</div>

### E6 · Decaimiento: autocorrelación móvil

In [ ]:
print(inspect.getsource(rolling_autocorr))

decay_returns = np.log(markets['Decaimiento']).diff()
rolling = rolling_autocorr(decay_returns, 200, 1)
cut_time = markets['Decaimiento'].index[int(N * TRAIN_FRACTION)]
fig, ax = plt.subplots()
ax.plot(rolling.index, rolling, color='#31688e', label='ACF(1) móvil, 200 retornos (sin centrar)')
ax.axvline(cut_time, color='firebrick', ls='--', label='Corte (φ pasa a 0)')
ax.axhline(0, color='gray', lw=.8)
ax.set(title='Decaimiento: autocorrelación móvil', xlabel='Cierre UTC', ylabel='ACF(1)'); ax.legend(fontsize=8)
fig.autofmt_xdate(); plt.show()
display(pd.DataFrame({th: regime_detection_delay(rolling, cut_time, th) for th in (.2, .1)}).T)
print(f"ACF(1) móvil justo en el corte: {rolling.loc[cut_time]:.3f} | 200 velas después: {rolling.loc[cut_time:].iloc[200]:.3f}")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · El cambio de régimen se ve tarde</b>

- En el corte (02:01 del 2 de enero), la autocorrelación móvil todavía vale **0,33**: la ventana de 200 retornos está llena de datos del régimen persistente.
- Tarda **84 velas** en caer por debajo de 0,2 y **101 velas** por debajo de 0,1. Solo cuando la ventana se ha renovado entera (200 velas después) refleja el régimen nuevo: −0,13 en ese punto.
- **No se detecta inmediatamente.** Una ventana hacia atrás siempre reacciona con retraso, y una ventana más corta reaccionaría antes, pero con mucho más ruido (ver P04, E3). Durante ese retraso, una estrategia basada en la señal seguiría operando sin ella.
- Los umbrales de 0,2 y 0,1 son ilustrativos. No se usaron para cambiar nada del test ya observado.

</div>

### E7 · El coste puede borrar un patrón real

In [ ]:
e7 = pd.concat([trade_experiment(markets['Persistente'], c, TRAIN_FRACTION).assign(cost_bps=c) for c in (0, 5, 10, 20, 40)])
display(e7.pivot(index='cost_bps', columns='periodo', values='net_bps').round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Estadísticamente real, económicamente inútil</b>

Control persistente (autocorrelación 0,34 en validación, p ≈ 0):

| Coste | Neto en desarrollo | Neto en validación |
|---|---|---|
| 0 bps | +0,72 bps | **+0,19 bps** |
| 5 bps | −4,28 bps | −4,81 bps |
| 10 bps | −9,28 bps | −9,81 bps |
| 40 bps | −39,28 bps | −39,81 bps |

- **Sí, un patrón estadístico puede desaparecer económicamente.** La dependencia es indiscutible (p ≈ 0), pero la ganancia bruta por operación es de **0,19 bps**. Con 5 bps de coste ya se pierde.
- ¿Por qué tan poco con una autocorrelación de 0,34? Porque la entrada se retrasa una vela: se compra en t + 1 y se cobra el retorno de t + 2, que depende de r_t solo a través de φ² ≈ 0,12. Es el mismo mecanismo que se vio en P05.
- **Coste hipotético frente a medido:** aquí 20 bps es un número fijo inventado para el ejemplo. Un coste medido con P08 sale del libro real en el instante de cada operación (spread, profundidad para el tamaño elegido, comisión real del exchange), y cambia de una operación a otra. Con un coste medido, la pregunta deja de ser "¿gana con 20 bps?" y pasa a ser "¿gana con lo que habría costado de verdad?".

</div>

### E8 · Datos reales de Kraken
No hay un `series.csv` de P04 exportado en el repositorio. Usamos las últimas 720 velas de 1 minuto de Kraken BTC/EUR (descarga de P05). Cambia `LIVE_E8` a `True` para repetirlo; para usar un CSV propio de P04, cambia `DATA_MODE` a `'csv'` en la configuración.

In [ ]:
LIVE_E8 = False

if LIVE_E8:
    from quantlab.p05.signals import fetch_kraken_ohlc, regularize
    live_close, live_prov = fetch_kraken_ohlc('XBTEUR', 1, WINDOW)  # si falla, lanza el error
    live_close = regularize(live_close, WINDOW)
    print(live_prov['received_at'], '|', live_close.index.min(), '—', live_close.index.max(),
          '| velas:', len(live_close), '| huecos:', int(live_close.isna().sum()))
    live_diag, live_acf = diagnostics({'Kraken': live_close}, LAGS, TRAIN_FRACTION)
    display(live_diag.round(4))
    live_r = np.log(live_close).diff().dropna()
    print("Ljung–Box sobre |r| (volatilidad):"); display(ljung_box_table(live_r.abs(), (5, 10)).round(4))
    display(pd.concat([trade_experiment(live_close, c, TRAIN_FRACTION).assign(cost_bps=c) for c in (0, ROUND_TRIP_BPS)]).round(3))
else:
    print("LIVE_E8 = False: no se consulta Kraken. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Kraken BTC/EUR: sin dependencia lineal detectable; evidencia insuficiente</b>

**Procedencia:** endpoint público OHLC de Kraken, BTC/EUR, velas de 1 minuto. Recepción 19-11-2026 21:05:33 UTC. **Cobertura:** 720 velas, de 09:06 a 21:05 UTC (12 horas), **0 huecos**. Desarrollo: 467 retornos; validación: 251.

| | ACF(1) | Ljung–Box (5) | Ljung–Box (10) |
|---|---|---|---|
| Desarrollo | −0,025 | p = 0,29 | p = 0,29 |
| Validación | +0,046 | p = 0,54 | p = 0,52 |

- **Magnitud del efecto:** las autocorrelaciones van de −0,10 a +0,12 en los 10 retardos, y cambian de signo entre periodos. Ljung–Box no rechaza en ningún caso: **no hay dependencia lineal detectable**.
- **Pero eso no es independencia:** la autocorrelación del **valor absoluto** de los retornos es **+0,24**, y Ljung–Box sobre |r| la rechaza con p ≈ 0. Los movimientos grandes vienen en grupos (agrupamiento de volatilidad), aunque su dirección no se pueda anticipar. Es justo la advertencia del cuaderno: la ausencia de autocorrelación no demuestra independencia.
- **Costes:** la regla de persistencia ganó −0,36 bps (desarrollo) y −1,10 bps (validación) **en bruto**, con 110 y 49 operaciones. Con 20 bps, −20,4 y −21,1 bps.
- **Limitaciones:** son 12 horas de un solo día; con 251 retornos de validación, el error estándar de una autocorrelación es de ≈ 0,06, mayor que cualquier efecto plausible. Además, la heterocedasticidad visible en |r| hace que los p-valores clásicos de Ljung–Box no estén bien calibrados.

**Conclusión: evidencia insuficiente.** No se puede afirmar ni negar una ineficiencia de BTC/EUR con estos datos. Haría falta un histórico de meses, contrastes robustos a la heterocedasticidad y costes medidos con P08.

</div>

## 9. Integración y trazabilidad
Exportación opcional. Los resultados de simulación se identifican como tales. El CSV de señales contiene resultados de validación ya observados: no lo utilices para seleccionar nuevas señales y volver a llamarlas fuera de muestra.

In [ ]:
if EXPORT:
    folder=Path('quantlab_exports'); folder.mkdir(exist_ok=True)
    stamp=pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    prefix=folder/f'p11_{DATA_MODE}_{stamp}'
    diagnostics_table.to_csv(str(prefix)+'_diagnostics.csv',index=False)
    signal_tests.to_csv(str(prefix)+'_null_signals.csv',index=False)
    performance.to_csv(str(prefix)+'_performance.csv',index=False)
    meta={'mode':DATA_MODE,'csv':CSV_PATH if DATA_MODE=='csv' else None,'seed':SEED,'window':WINDOW,'train_fraction':TRAIN_FRACTION,'lags':LAGS,'alpha':ALPHA,'n_signals':N_SIGNALS,'n_simulations':N_SIMULATIONS,'cost_bps':ROUND_TRIP_BPS,'entry_delay_bars':1,'holding_bars':1,'synthetic_controls':True,'gap_policy':'no_fill'}
    Path(str(prefix)+'_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
    print(prefix)
else:
    print('Exportación desactivada.')

## 10. Conclusiones y siguiente práctica
- La predictibilidad lineal, la significación estadística y el beneficio neto son conceptos distintos.
- La ausencia de autocorrelación no implica independencia completa ni demuestra eficiencia; puede haber dependencia en volatilidad u otras variables.
- Una hipótesis descubierta tras muchas pruebas debe corregirse y validarse con datos cronológicamente posteriores. Si modificas la hipótesis después de mirar el test, necesitas otro periodo intacto.
- Para datos reales con dependencia temporal profundizaremos en bootstrap por bloques, errores HAC, pruebas de estabilidad y múltiples comparaciones en módulos posteriores.

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ ¿QUÉ EVIDENCIA HARÍA FALTA PARA ACEPTAR UNA ANOMALÍA REAL?</b>

1. **Una hipótesis escrita antes de mirar los datos de evaluación**, con la señal, el horizonte, el umbral y los costes fijados de antemano, y el número de hipótesis probadas registrado.
2. **Significación corregida por pruebas múltiples** (Bonferroni o BH según el caso) y con contrastes robustos a la dependencia temporal y a la heterocedasticidad (bootstrap por bloques, errores HAC), no p-valores clásicos.
3. **Validación en un periodo posterior nunca usado**, con la selección congelada. El efecto debe mantener el signo y una magnitud parecida (no como la señal 105, que pasó de 0,076 a 0,021).
4. **Estabilidad en el tiempo:** que el efecto aparezca en varios subperiodos y regímenes, y no solo en un tramo (el mercado "Decaimiento" pasa los tests en desarrollo y desaparece justo después).
5. **Viabilidad económica después de costes medidos** (P08), no hipotéticos: el control persistente es estadísticamente indiscutible y económicamente inútil.
6. **Un mecanismo plausible** que explique por qué existiría y por qué otros participantes no lo han eliminado ya.

Con los datos de esta práctica, **ninguna** de las señales probadas cumple estos requisitos.

📄 Resumen en `docs/conclusiones/P11_market_efficiency.md`

</div>

**P12:** separación de alpha, beta y exposiciones a factores.